# convert sampled values into a dataset with lemma locations

sampled values are of the format [(project_dir, file, lemma_name)]
this notebook will convert them into lemma locations, and write them into a json file to be loaded

In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
from pathlib import Path
import typing as t
import re
from pprint import pprint
import coq_serapy as c
import json
import pickle

from src.config import CONFIG
from src.coq_serapy_util import LemmaLocation
from src.dataset import Example

# CoqGym

In [ ]:
DEV_SET_FILE = Path(CONFIG.ROOT_DIR) / "data/COQGYM_DEV_SAMPLED_DATASET_RAW.txt"
TEST_SET_FILE = Path(CONFIG.ROOT_DIR) / "data/COQGYM_TEST_SAMPLED_DATASET_RAW.txt"

print(DEV_SET_FILE)
print(TEST_SET_FILE)

## parse line in text file

In [ ]:
# ('coqgym/coq-projects/ruler-compass-geometry', 'C12_Angles_Opposes.v', 'CongruentOpposedStrictTriangles') -> 'coqgym/coq-projects/ruler-compass-geometry/C12_Angles_Opposes.v', 'C12_Angles_Opposes', 'CongruentOpposedStrictTriangles'
# ('coqgym/coq-projects/VST', 'progs/verif_peel.v', 'body_f') -> 'coqgym/coq-projects/VST', 'progs/verif_peel.v', 'body_f'
# ('coqgym/coq-projects/CompCert', 'flocq/Core/FLT.v', 'FLT_format_satisfies_any') -> 'coqgym/coq-projects/CompCert', 'flocq/Core/FLT.v', 'FLT_format_satisfies_any'

LINE_REGEX = r"^\('(.+)', '(.+)', '(.+)'\)$"
def parse_line(line: str) -> t.Tuple[str, str, str]:
    match = re.match(LINE_REGEX, line)
    if match is None:
        raise ValueError(f"Could not parse line: {line}")
    return match.groups()

pprint(parse_line("('coqgym/coq-projects/ruler-compass-geometry', 'C12_Angles_Opposes.v', 'CongruentOpposedStrictTriangles')"))

## figure out project coq version from project name

In [ ]:
COQGYM_TEST_SET_FILE = current_directory / "coqgym_test_set.json"

with open(COQGYM_TEST_SET_FILE, "r") as f:
    test_set = json.load(f)

pprint(test_set[:5])

In [ ]:
PROJECT_NAME_TO_COQ_VERSION = {
    obj["project_name"]: obj["switch"][len('coq-'):]
    for obj in test_set
}

pprint(PROJECT_NAME_TO_COQ_VERSION)

## convert tuple from text file into a lemmalocation

In [ ]:
section_regex = re.compile(r"Section\s+(\w+)")
end_section_regex = re.compile(r"End\s+(\w+)")
lemma_regex = re.compile(r"(Example|Lemma|Theorem)\s+(\w+)\s*:([\s\S]*)\.")

TEST_PROJECT_NAME = "coqgym/coq-projects/ruler-compass-geometry"
TEST_FILE_NAME = "C12_Angles_Opposes.v"
TEST_LEMMA_NAME = "CongruentOpposedStrictTriangles"

TEST_FILE_PATH = Path(CONFIG.ROOT_DIR) / Path(CONFIG.PROJECTS_ROOT) / TEST_PROJECT_NAME / TEST_FILE_NAME
print(TEST_FILE_PATH)

In [ ]:
commands = c.coq_util.load_commands(str(TEST_FILE_PATH.resolve()))
commands = [c.coq_util.kill_comments(command).strip() for command in commands]

pprint(commands[:10])

In [ ]:

section_names: t.List[str] = []
while len(commands) > 0:
    command = commands.pop(0)
    print("processing", command)
    
    section_match = section_regex.match(command)
    if section_match is not None:
        print("section match")
        section_name = section_match.group(1)
        section_names.append(section_name)
        continue

    end_section_match = end_section_regex.match(command)
    if end_section_match is not None:
        print("end section match")
        section_name = end_section_match.group(1)
        if section_name != section_names[-1]:
            raise ValueError(f"Section name mismatch: {section_name} != {section_names[-1]}")
        section_names.pop()
        continue

    lemma_match = lemma_regex.match(command)
    if lemma_match is not None:
        print("lemma match")
        lemma_name = lemma_match.group(2)
        print(lemma_name)
        if lemma_name == TEST_LEMMA_NAME:
            print("found lemma", section_names)
            break

In [ ]:
def compute_example(project_name: str, file_name: str, lemma_name: str) -> t.Optional[Example]:
    file_path = Path(CONFIG.ROOT_DIR) / Path(CONFIG.PROJECTS_ROOT) / project_name / file_name
    commands = c.coq_util.load_commands(str(file_path.resolve()))
    commands = [c.coq_util.kill_comments(command).strip() for command in commands]

    section_names: t.List[str] = []
    while len(commands) > 0:
        command = commands.pop(0)
        section_match = section_regex.match(command)
        if section_match is not None:
            print(f"{command} matches section")
            section_name = section_match.group(1)
            section_names.append(section_name)
            continue

        end_section_match = end_section_regex.match(command)
        if end_section_match is not None:
            print(f"{command} matches end section")
            section_name = end_section_match.group(1)
            if len(section_names) == 0 or section_name != section_names[-1]:
                print(f"Section name mismatch: {section_name}")
                continue
            section_names.pop()
            continue

        lemma_match = lemma_regex.match(command)
        if lemma_match is not None:
            other_lemma_name = lemma_match.group(2)
            if lemma_name == other_lemma_name:
                actual_project_name = project_name.split("/")[-1]
                coq_version = PROJECT_NAME_TO_COQ_VERSION.get(actual_project_name, "8.12")
                lemma_location = LemmaLocation(
                    project_name=project_name,
                    file_name=file_name,
                    lemma_name=other_lemma_name,
                    section_names=section_names,
                    coq_version=coq_version
                )
                example = Example(
                    gold_standard_proof="",
                    proposition_command=command,
                    location=lemma_location
                )
                return example
                    
    return None
    

In [ ]:
pprint(compute_example(TEST_PROJECT_NAME, TEST_FILE_NAME, TEST_LEMMA_NAME))

## putting it all together

In [ ]:
DEV_SET_LINES = DEV_SET_FILE.read_text().strip().split("\n")
TEST_SET_LINES = TEST_SET_FILE.read_text().strip().split("\n")

pprint(DEV_SET_LINES[:5])
pprint(TEST_SET_LINES[:5])

In [ ]:
PARSED_DEV_SET = [parse_line(line) for line in DEV_SET_LINES]
PARSED_TEST_SET = [parse_line(line) for line in TEST_SET_LINES]

pprint(PARSED_DEV_SET[:5])
pprint(PARSED_TEST_SET[:5])

In [ ]:
DEV_EXAMPLES = []
for project_name, file_name, lemma_name in PARSED_DEV_SET:
    print(project_name, file_name, lemma_name)
    example = compute_example(project_name, file_name, lemma_name)
    pprint(example)
    DEV_EXAMPLES.append(example)

pprint(DEV_EXAMPLES[:5])
# count how many nones are in the dev set
print(len([loc for loc in DEV_EXAMPLES if loc is None]))

In [ ]:
TEST_EXAMPLES = []
for project_name, file_name, lemma_name in PARSED_TEST_SET:
    print(project_name, file_name, lemma_name)
    example = compute_example(project_name, file_name, lemma_name)
    pprint(example)
    TEST_EXAMPLES.append(example)

pprint(TEST_EXAMPLES[:5])
pprint(len([loc for loc in TEST_EXAMPLES if loc is None]))

In [ ]:
# pickle the dev set to ./COQGYM_DEV_SAMPLED.pkl
# pickle the test set to ./COQGYM_TEST_SAMPLED.pkl
DEV_SAMPLED_FILE = current_directory / "../../data/COQGYM_DEV_SAMPLED_DATASET.pkl"
TEST_SAMPLED_FILE = current_directory / "../../data/COQGYM_TEST_SAMPLED_DATASET.pkl"

with open(DEV_SAMPLED_FILE, "wb") as f:
    pickle.dump(DEV_EXAMPLES, f)

with open(TEST_SAMPLED_FILE, "wb") as f:
    pickle.dump(TEST_EXAMPLES, f)

# coq-wigderson

In [ ]:
LINE_REGEX = r"^(.+), (.+)$"
def parse_line(line: str) -> t.Tuple[str, str]:
    match = re.match(LINE_REGEX, line)
    if match is None:
        raise ValueError(f"Could not parse line: {line}")
    return match.groups()

pprint(parse_line("coloring.v, subgraph_coloring_ok"))

In [ ]:
W_DEV_SET_FILE = Path(CONFIG.ROOT_DIR) / "data/COQ_WIGDERSON_DEV_SAMPLED_DATASET_RAW.txt"
print(W_DEV_SET_FILE)

In [ ]:
section_regex = re.compile(r"Section\s+(\w+)")
end_section_regex = re.compile(r"End\s+(\w+)")
lemma_regex = re.compile(r"(Example|Lemma|Theorem)\s+(\w+).")
# lemma_regex = re.compile(r"(Example|Lemma|Theorem)\s+(\w+)\s*(\{\w+\})?\s*:([\s\S]*)\.")

In [ ]:
inputs = [
    'Lemma Munion_in {A} : forall i (m1 m2 : M.t A),\n    M.In i (Munion m1 m2) <-> M.In i m1 \\/ M.In i m2.',
    'Lemma two_color_up_inj f g (inj : S.elt -> S.elt) :\n  injective inj ->\n  undirected g ->\n  coloring_ok two_colors g f ->\n  {h | coloring_ok (fold_right S.add S.empty [inj 1;inj 2]) g h}.'
]

for input in inputs:
    lemma_match = lemma_regex.match(input)
    print(input)
    print(lemma_match)
    if lemma_match is not None:
        print(lemma_match.groups())
    print()

In [ ]:
PROJECT_NAME='coq-wigderson'
PROJECT_NAME_TO_COQ_VERSION = {
    'coq-wigderson': '8.13'
}

In [ ]:
def compute_example(project_name: str, file_name: str, lemma_name: str) -> t.Optional[Example]:
    file_path = Path(CONFIG.ROOT_DIR) / Path(CONFIG.PROJECTS_ROOT) / project_name / file_name
    commands = c.coq_util.load_commands(str(file_path.resolve()))
    commands = [c.coq_util.kill_comments(command).strip() for command in commands]

    section_names: t.List[str] = []
    while len(commands) > 0:
        command = commands.pop(0)
        section_match = section_regex.match(command)
        if section_match is not None:
            print(f"{command} matches section")
            section_name = section_match.group(1)
            section_names.append(section_name)
            continue

        end_section_match = end_section_regex.match(command)
        if end_section_match is not None:
            print(f"{command} matches end section")
            section_name = end_section_match.group(1)
            if len(section_names) == 0 or section_name != section_names[-1]:
                print(f"Section name mismatch: {section_name}")
                continue
            section_names.pop()
            continue

        lemma_match = lemma_regex.match(command)
        # print(command, 'lemma_match', lemma_match)
        if lemma_match is not None:
            other_lemma_name = lemma_match.group(2)
            if lemma_name == other_lemma_name:
                actual_project_name = project_name.split("/")[-1]
                coq_version = PROJECT_NAME_TO_COQ_VERSION.get(actual_project_name, "8.12")
                lemma_location = LemmaLocation(
                    project_name=project_name,
                    file_name=file_name,
                    lemma_name=other_lemma_name,
                    section_names=section_names,
                    coq_version=coq_version
                )
                example = Example(
                    gold_standard_proof="",
                    proposition_command=command,
                    location=lemma_location
                )
                return example
                    
    return None
    

In [ ]:
inputs = [
    ('coloring.v', 'subgraph_coloring_ok'),
    ('munion.v', 'Munion_case'),
    ('wigderson.v', 'two_color_up_inj')
]

for input in inputs:
    print(input)
    pprint(compute_example(PROJECT_NAME, *input))
    print()

In [ ]:
W_DEV_LINES = W_DEV_SET_FILE.read_text().strip().split("\n")
pprint(W_DEV_LINES[:5])

In [ ]:
PARSED_W_DEV_SET = [parse_line(line) for line in W_DEV_LINES]
pprint(PARSED_W_DEV_SET[:5])

In [ ]:
W_DEV_EXAMPLES = []
for file_name, lemma_name in PARSED_W_DEV_SET:
    print(file_name, lemma_name)
    example = compute_example(PROJECT_NAME, file_name, lemma_name)
    pprint(example)
    W_DEV_EXAMPLES.append(example)

pprint(W_DEV_EXAMPLES[:5])
# count how many nones are in the dev set
print(len([loc for loc in W_DEV_EXAMPLES if loc is None]))

In [ ]:
# pickle the dev set to ./COQ_WIGDERSON_DEV_SAMPLED.pkl
W_DEV_SAMPLED_FILE = current_directory / "../../data/COQ_WIGDERSON_DEV_SAMPLED_DATASET.pkl"
with open(W_DEV_SAMPLED_FILE, "wb") as f:
    pickle.dump(W_DEV_EXAMPLES, f)